In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Configuration
MODEL_CHECKPOINT = "./bart_large_medical_output"  # Path to your fine-tuned model
INPUT_TEXT = "This is a complex medical sentence that needs to be simplified for better understanding by the general public."  # Example input
PREFIX = "simplify: "
MAX_LENGTH = 256

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# Prepare input
input_text = PREFIX + INPUT_TEXT
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate simplified output
with torch.no_grad():
    outputs = model.generate(input_ids, max_length=MAX_LENGTH)

# Decode output
simplified_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print results
print("Original Text:", INPUT_TEXT)
print("Simplified Text:", simplified_text)

#Example with batch inference.
batch_input = [
    "This is a complex medical sentence that needs to be simplified for better undesrstanding by the general public.",
    "The patient presented with symptoms indicative of a severe respiratory infection, necessitating immediate medical intervention.",
    "Administering the medication intravenously ensures rapid absorption and prompt therapeutic effects.",
]
batch_input = [PREFIX + text for text in batch_input]

batch_input_ids = tokenizer(batch_input, return_tensors="pt", padding=True).input_ids

with torch.no_grad():
    batch_outputs = model.generate(batch_input_ids, max_length=MAX_LENGTH)

simplified_batch_text = tokenizer.batch_decode(batch_outputs, skip_special_tokens=True)

print("\nBatch Inference Results:")
for i, simplified_text_single in enumerate(simplified_batch_text):
    print(f"Original Text: {batch_input[i][len(PREFIX):]}")
    print(f"Simplified Text: {simplified_text_single}")

import error: No module named 'triton'
Original Text: This is a complex medical sentence that needs to be simplified for better understanding by the general public.
Simplified Text: This is a complex medical sentence that needs to be simplified for better understanding by the general public.

Batch Inference Results:
Original Text: This is a complex medical sentence that needs to be simplified for better undesrstanding by the general public.
Simplified Text: This is a complex medical sentence that needs to be simplified for better undesrstanding by the general public.
Original Text: The patient presented with symptoms indicative of a severe respiratory infection, necessitating immediate medical intervention.
Simplified Text: The patient presented with symptoms indicative of a severe respiratory infection, necessitating immediate medical intervention.
Original Text: Administering the medication intravenously ensures rapid absorption and prompt therapeutic effects.
Simplified Text: The a

In [4]:
import nltk
nltk.jaccard_distance()

TypeError: jaccard_distance() missing 2 required positional arguments: 'label1' and 'label2'

In [13]:
from __future__ import division
from collections import Counter
import glob
import numpy as np
from argparse import ArgumentParser


"""
Based on code by Wei Xu
#https://github.com/cocoxu/simplification

This a slightly improved version of SARI implementation to exclude the spurious ngrams while 
calculating the F1 score for add. For example, if the input contains the phrase “is very beautiful”, 
the phrase “is beautiful” is treated as a new phrase in the original implementation even though it is 
caused by the delete operation.

"""

def is_subsequence(str1,str2):
    m = len(str1) 
    n = len(str2)
    i, j = 0, 0
    while j<m and i<n: 
        if str1[j] == str2[i]:     
            j = j+1    
        i = i + 1
    return j==m 

def SARIngram(sgrams, cgrams, rgramslist, numref, complex):
    rgramsall = [rgram for rgrams in rgramslist for rgram in rgrams]
    rgramcounter = Counter(rgramsall)

    sgramcounter = Counter(sgrams)
    sgramcounter_rep = Counter()
    for sgram, scount in sgramcounter.items():
        sgramcounter_rep[sgram] = scount * numref

    cgramcounter = Counter(cgrams)
    cgramcounter_rep = Counter()
    for cgram, ccount in cgramcounter.items():
        cgramcounter_rep[cgram] = ccount * numref

    # KEEP
    keepgramcounter_rep = sgramcounter_rep & cgramcounter_rep
    keepgramcountergood_rep = keepgramcounter_rep & rgramcounter
    keepgramcounterall_rep = sgramcounter_rep & rgramcounter

    keeptmpscore1 = 0
    keeptmpscore2 = 0
    for keepgram in keepgramcountergood_rep:
        keeptmpscore1 += keepgramcountergood_rep[keepgram] / keepgramcounter_rep[keepgram]
        keeptmpscore2 += keepgramcountergood_rep[keepgram] / keepgramcounterall_rep[keepgram]
        # print "KEEP", keepgram, keepscore, cgramcounter[keepgram], sgramcounter[keepgram], rgramcounter[keepgram]
    keepscore_precision = 0
    if len(keepgramcounter_rep) > 0:
        keepscore_precision = keeptmpscore1 / len(keepgramcounter_rep)
    keepscore_recall = 0
    if len(keepgramcounterall_rep) > 0:
        keepscore_recall = keeptmpscore2 / len(keepgramcounterall_rep)
    keepscore = 0
    if keepscore_precision > 0 or keepscore_recall > 0:
        keepscore = 2 * keepscore_precision * keepscore_recall / (keepscore_precision + keepscore_recall)

    # DELETION
    delgramcounter_rep = sgramcounter_rep - cgramcounter_rep
    delgramcountergood_rep = delgramcounter_rep - rgramcounter
    delgramcounterall_rep = sgramcounter_rep - rgramcounter
    deltmpscore1 = 0
    deltmpscore2 = 0
    for delgram in delgramcountergood_rep:
        deltmpscore1 += delgramcountergood_rep[delgram] / delgramcounter_rep[delgram]
        deltmpscore2 += delgramcountergood_rep[delgram] / delgramcounterall_rep[delgram]
    delscore_precision = 0
    if len(delgramcounter_rep) > 0:
        delscore_precision = deltmpscore1 / len(delgramcounter_rep)
    delscore_recall = 0
    if len(delgramcounterall_rep) > 0:
        delscore_recall = deltmpscore1 / len(delgramcounterall_rep)
    delscore = 0
    if delscore_precision > 0 or delscore_recall > 0:
        delscore = 2 * delscore_precision * delscore_recall / (delscore_precision + delscore_recall)

    # ADDITION
    addgramcounter = set(cgramcounter) - set(sgramcounter)
    addgramcountergood = set(addgramcounter) & set(rgramcounter)
    addgramcounterall = set(rgramcounter) - set(sgramcounter)

    sgrams_set = set()
    for gram in sgrams:
        sgrams_set.update(gram.split())

    addgramcountergood_new = set()
    for gram in addgramcountergood:
        if any([tok not in sgrams_set for tok in gram.split()]) or not is_subsequence(gram.split(), complex.split()):
            addgramcountergood_new.add(gram)
    addgramcountergood = addgramcountergood_new

    addtmpscore = 0
    for _ in addgramcountergood:
        addtmpscore += 1

    addscore_precision = 0
    addscore_recall = 0
    if len(addgramcounter) > 0:
        addscore_precision = addtmpscore / len(addgramcounter)
    if len(addgramcounterall) > 0:
        addscore_recall = addtmpscore / len(addgramcounterall)
    addscore = 0
    if addscore_precision > 0 or addscore_recall > 0:
        addscore = 2 * addscore_precision * addscore_recall / (addscore_precision + addscore_recall)

    return (keepscore, (delscore_precision, delscore_recall, delscore), addscore)


def SARIsent(ssent, csent, rsents):
    numref = len(rsents)

    s1grams = ssent.lower().split(" ")
    c1grams = csent.lower().split(" ")
    s2grams = []
    c2grams = []
    s3grams = []
    c3grams = []
    s4grams = []
    c4grams = []

    r1gramslist = []
    r2gramslist = []
    r3gramslist = []
    r4gramslist = []

    for rsent in rsents:
        r1grams = rsent.lower().split(" ")
        r2grams = []
        r3grams = []
        r4grams = []
        r1gramslist.append(r1grams)
        for i in range(0, len(r1grams) - 1):
            if i < len(r1grams) - 1:
                r2gram = r1grams[i] + " " + r1grams[i + 1]
                r2grams.append(r2gram)
            if i < len(r1grams) - 2:
                r3gram = r1grams[i] + " " + r1grams[i + 1] + " " + r1grams[i + 2]
                r3grams.append(r3gram)
            if i < len(r1grams) - 3:
                r4gram = r1grams[i] + " " + r1grams[i + 1] + " " + r1grams[i + 2] + " " + r1grams[i + 3]
                r4grams.append(r4gram)
        r2gramslist.append(r2grams)
        r3gramslist.append(r3grams)
        r4gramslist.append(r4grams)

    for i in range(0, len(s1grams) - 1):
        if i < len(s1grams) - 1:
            s2gram = s1grams[i] + " " + s1grams[i + 1]
            s2grams.append(s2gram)
        if i < len(s1grams) - 2:
            s3gram = s1grams[i] + " " + s1grams[i + 1] + " " + s1grams[i + 2]
            s3grams.append(s3gram)
        if i < len(s1grams) - 3:
            s4gram = s1grams[i] + " " + s1grams[i + 1] + " " + s1grams[i + 2] + " " + s1grams[i + 3]
            s4grams.append(s4gram)

    for i in range(0, len(c1grams) - 1):
        if i < len(c1grams) - 1:
            c2gram = c1grams[i] + " " + c1grams[i + 1]
            c2grams.append(c2gram)
        if i < len(c1grams) - 2:
            c3gram = c1grams[i] + " " + c1grams[i + 1] + " " + c1grams[i + 2]
            c3grams.append(c3gram)
        if i < len(c1grams) - 3:
            c4gram = c1grams[i] + " " + c1grams[i + 1] + " " + c1grams[i + 2] + " " + c1grams[i + 3]
            c4grams.append(c4gram)

    (keep1score, del1score, add1score) = SARIngram(s1grams, c1grams, r1gramslist, numref, ssent)
    (keep2score, del2score, add2score) = SARIngram(s2grams, c2grams, r2gramslist, numref, ssent)
    (keep3score, del3score, add3score) = SARIngram(s3grams, c3grams, r3gramslist, numref, ssent)
    (keep4score, del4score, add4score) = SARIngram(s4grams, c4grams, r4gramslist, numref, ssent)

    del1p, del1r, del1f = del1score
    del2p, del2r, del2f = del2score
    del3p, del3r, del3f = del3score
    del4p, del4r, del4f = del4score

    avgkeepscore = sum([keep1score, keep2score, keep3score, keep4score]) / 4
    avgdelpscore = sum([del1p, del2p, del3p, del4p]) / 4
    avgdelrscore = sum([del1r, del2r, del3r, del4r]) / 4
    avgdelfscore = sum([del1f, del2f, del3f, del4f]) / 4
    avgaddscore = sum([add1score, add2score, add3score, add4score]) / 4
    finalpscore = (avgkeepscore + avgdelpscore + avgaddscore) / 3
    finalfscore = (avgkeepscore + avgdelfscore + avgaddscore) / 3
    return avgkeepscore, (avgdelpscore, avgdelrscore, avgdelfscore), avgaddscore, (finalpscore, finalfscore)


def compute_sari(complex_sentences, reference_sentences, simplified_sentences):

    delp_scores = list()
    delr_scores = list()
    delf_scores = list()
    add_scores = list()
    sari_scores = list()
    sarif_scores = list()
    keep_scores = list()
    for i in range(len(simplified_sentences)):
        keep, dels, add, final = SARIsent(complex_sentences[i], simplified_sentences[i],
                                          reference_sentences[i])
        add_scores.append(add)
        delp_scores.append(dels[0])
        delr_scores.append(dels[1])
        delf_scores.append(dels[2])
        keep_scores.append(keep)
        sari_scores.append(final[0])
        sarif_scores.append(final[1])
    
    return np.mean(sari_scores), np.mean(sarif_scores), np.mean(add_scores), np.mean(keep_scores), np.mean(
        delp_scores), np.mean(delr_scores), np.mean(delf_scores)



if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument("-c", "--complex", dest="complex",
                        help="complex sentences", metavar="FILE")
    parser.add_argument("-r", "--reference", dest="reference",
                        help="reference sentences", metavar="FILE")
    parser.add_argument("-s", "--simplified", dest="simplified",
                        help="simplified sentences", metavar="FILE")

    args = parser.parse_args()
    print('SARI score: {}'.format(compute_sari(args.complex, args.reference, args.simplified)))

usage: ipykernel_launcher.py [-h] [-c FILE] [-r FILE] [-s FILE]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Ruben\AppData\Roaming\jupyter\runtime\kernel-v33bfc9d1ff8041ae3b2ea0f29fa3d55ac0967f246.json


SystemExit: 2

c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\IPython\core\interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
import evaluate
import nltk
from nltk.metrics import jaccard_distance
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import torch
import numpy as np
import random
import logging


sources = [
    "The patient presented with a history of recurrent headaches.",
    "This is a complex medical term that needs explanation.",
    "The study investigated the efficacy of a novel therapeutic intervention.",
]
predictions = [
    "The patient had headaches again.",
    "This term is hard to understand.",
    "The study looked at how well the treatment worked.",
]
references = [
    ["The patient had repeated headaches."],
    ["This medical term is hard to understand."],
    ["The study checked if the new treatment helped."],
]

    

In [10]:
sari_metric = evaluate.load("sari")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
results = {}  # Store results


sari_results = sari_metric.compute(sources=sources, predictions=predictions, references=references)

 

In [11]:
sari_results

{'sari': 73.88217492384157}

In [ ]:
results["sari_substitution"] = sari_results["sari_Ngram_F1_p"][:, 1].mean()


In [ ]:
   jaccard_scores = []
    semantic_similarities = []
    diff_words_percentages = []

    for source, pred, ref in zip(sources, predictions, references):
        source_words = set(source.lower().split())
        pred_words = set(pred.lower().split())
        jaccard_scores.append(1 - jaccard_distance(source_words, pred_words))
        diff_words_percentages.append(
            (len(source_words.symmetric_difference(pred_words)) / len(source_words.union(pred_words))) * 100
        )
        source_embedding = sentence_model.encode(source, convert_to_tensor=True)
        pred_embedding = sentence_model.encode(pred, convert_to_tensor=True)
        semantic_similarities.append(F.cosine_similarity(source_embedding, pred_embedding).item())

        print("\n--- Example ---")
        print(f"Source: {source}")
        print(f"Prediction: {pred}")
        print(f"Reference: {ref}")
        print(f"Jaccard: {jaccard_scores[-1]:.4f}")
        print(f"Semantic Similarity: {semantic_similarities[-1]:.4f}")
        print(f"Different Words (%): {diff_words_percentages[-1]:.2f}%")

    results["jaccard"] = np.mean(jaccard_scores)
    results["semantic_similarity"] = np.mean(semantic_similarities)
    results["diff_words"] = np.mean(diff_words_percentages)


metrics = calculate_metrics_on_batch(sources, predictions, references)

print("\n--- Overall Metrics ---")
print(metrics)

In [20]:
import torch
from torch.utils.data import DataLoader
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    set_seed,
    EarlyStoppingCallback
)
import numpy as np
import evaluate
import optuna
from datasets import load_dataset
import nltk
import os
import multiprocessing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_path = r"C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\code\data\multiCochrane_all\unfiltered (r=0)\en"
data_files = {
    "train": os.path.join(base_path, "train0_en.csv"),
    "test": os.path.join(base_path, "test0_en.csv"),
    "validation": os.path.join(base_path, "val0_en.csv")
}
multi_cochrane_dataset = load_dataset("csv", data_files=data_files)

# Log dataset sizes
print("Dataset sizes:")
for split, ds in multi_cochrane_dataset.items():
    print(f"{split}: {len(ds)} examples")

# Add prefix to inputs
prefix = "simplify: "
def add_prefix(example):
    example['prefix'] = prefix
    return example

for split in multi_cochrane_dataset:
    multi_cochrane_dataset[split] = multi_cochrane_dataset[split].map(add_prefix)

# Tokenizer and preprocessing
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-large")
max_length = 128

def preprocess_examples(examples):
    inputs = [p + inp for p, inp in zip(examples['prefix'], examples['input_text'])]
    model_inputs = tokenizer(inputs, max_length=max_length, padding="max_length", truncation=True)
    labels = tokenizer(examples['target_text'], max_length=max_length, padding="max_length", truncation=True).input_ids
    labels = [[label if label != 0 else -100 for label in lbl] for lbl in labels]
    model_inputs["labels"] = labels
    return model_inputs

encoded_datasets = {
    split: ds.map(preprocess_examples, batched=True, remove_columns=ds.column_names)
    for split, ds in multi_cochrane_dataset.items()
}

final_output_dir = "./final_t5_small_simplification_antiparrot_fast/"
try:
    print(f"Loading final model from {final_output_dir} for testing...")
    model_for_testing = T5ForConditionalGeneration.from_pretrained(final_output_dir).to(device)
    tokenizer_for_testing = T5Tokenizer.from_pretrained(final_output_dir)
    model_for_testing.eval()
except Exception as e: print(f"Error loading final model for testing: {e}")

if "test" in multi_cochrane_dataset:
    num_test_examples = 20
    print(f"Generating simplifications for {num_test_examples} test examples...")
    test_set_original = multi_cochrane_dataset['test']

    for i in range(min(num_test_examples, len(test_set_original))):
        # Access original 'Expert' text via the renamed 'input_text' column
        original_text = test_set_original[i + 300]['input_text']
        if not original_text: continue

        prompt = f"{"Convert this complex text to simple language: "}{original_text}"
        try:
            inputs = tokenizer_for_testing(prompt, return_tensors="pt", max_length=max_length, truncation=True).to(device)
            # Use same generation params as before
            generated_ids = model_for_testing.generate(
                **inputs, max_length=max_length, do_sample=True, temperature=0.7, #top_p=0.9,
                num_return_sequences=1, #no_repeat_ngram_size=3, encoder_no_repeat_ngram_size=3
            )
            simplified = tokenizer_for_testing.decode(generated_ids[0], skip_special_tokens=True)

            print(f"\n--- Test Example {i+1} ---")
            print(f"Original: {original_text}")
            print(f"Model Simplified: {simplified}")
            original_words = set(original_text.lower().split()); simplified_words = set(simplified.lower().split())
            if original_words and simplified_words:
                    intersection = original_words.intersection(simplified_words); union = original_words.union(simplified_words)
                    jaccard = len(intersection) / len(union) if union else 0; print(f"Word Jaccard Similarity: {jaccard:.2f}")
            print("-" * 20)
        except Exception as test_e: print(f"Error testing example {i+1}: {test_e}")
else: print("No 'test' split found in the original medical dataset.")

Dataset sizes:
train: 61194 examples
validation: 83 examples
test: 395 examples
Loading final model from ./final_t5_small_simplification_antiparrot_fast/ for testing...
Generating simplifications for 20 test examples...

--- Test Example 1 ---
Original: Eight RCTs with 733 women in total that compared brief co-incubation and the standard insemination protocol were included.
Model Simplified: eighth total female trials with 733 women in total.
Word Jaccard Similarity: 0.23
--------------------

--- Test Example 2 ---
Original: Anabolic steroids are considered because of their ability to stimulate protein synthesis and build muscle mass.
Model Simplified: anabolic steroids are thought to be because they can stimulate protein synthesis and build muscle mass.
Word Jaccard Similarity: 0.60
--------------------

--- Test Example 3 ---
Original: Smoking prevalence in Indigenous youth is twice that of the non-Indigenous population, with tobacco experimentation commencing at an early age.
Model